# 03 · 同调相同，上同调环不同

**可运行实验**：构造有限三角化环面与 $S^1\vee S^1\vee S^2$，在 $\mathbb F_2$ 上计算上同调和杯积。它们均有 Betti 数 (1,2,1)，杯积却不同。对应 Q1、Q2、K1。

**理论推导**：一般杯积的良定义、同伦不变性见 Q1。本 Notebook 的断言只证明这些指定有限复形的计算输出，并非一般分类定理。另用 RP² 的胞腔矩阵说明整数挠与域系数之别。

依赖 NumPy，无网络与随机数。代码 MIT，原创说明 CC BY 4.0。

In [1]:
import itertools
import numpy as np

def closure(maximal):
    return sorted({tuple(sorted(s)) for face in maximal
                   for k in range(1, len(face)+1)
                   for s in itertools.combinations(face,k)}, key=lambda s:(len(s),s))

def boundary(cells, k):
    rows=[s for s in cells if len(s)==k]
    cols=[s for s in cells if len(s)==k+1]
    lookup={s:i for i,s in enumerate(rows)}
    out=np.zeros((len(rows),len(cols)),dtype=np.int64)
    for j,s in enumerate(cols):
        for i in range(len(s)):
            out[lookup[s[:i]+s[i+1:]],j]=(-1)**i
    return out

def rref2(matrix):
    a=np.array(matrix,dtype=np.int64,copy=True)%2
    pivots=[]; row=0
    for col in range(a.shape[1]):
        hit=next((r for r in range(row,a.shape[0]) if a[r,col]),None)
        if hit is None: continue
        a[[row,hit]]=a[[hit,row]]
        for r in range(a.shape[0]):
            if r!=row and a[r,col]: a[r]^=a[row]
        pivots.append(col); row+=1
        if row==a.shape[0]: break
    return a,pivots

def rank2(a): return len(rref2(a)[1])

def kernel2(a):
    r,pivots=rref2(a); vectors=[]
    for free in (j for j in range(r.shape[1]) if j not in pivots):
        v=np.zeros(r.shape[1],dtype=np.int64); v[free]=1
        for i,p in enumerate(pivots): v[p]=r[i,free]
        vectors.append(v)
    return vectors

def h1_cochains(d1,d2):
    # Coboundaries are the columns of D1 transpose; cocycles ker(D2 transpose).
    span=d1.T.copy()%2; old_rank=rank2(span); representatives=[]
    for v in kernel2(d2.T):
        candidate=np.column_stack([span,v]); new_rank=rank2(candidate)
        if new_rank>old_rank:
            representatives.append(v); span=candidate; old_rank=new_rank
    return representatives

def inspect(cells, fundamental_faces):
    d1,d2=boundary(cells,1),boundary(cells,2)
    assert np.all(d1@d2==0) # signed integer identity, before reducing mod 2
    counts=[sum(len(s)==k+1 for s in cells) for k in range(3)]
    betti=(counts[0]-rank2(d1),counts[1]-rank2(d1)-rank2(d2),counts[2]-rank2(d2))
    assert counts[0]-counts[1]+counts[2]==betti[0]-betti[1]+betti[2]
    h=h1_cochains(d1,d2)
    assert len(h)==betti[1]
    edges={s:i for i,s in enumerate(s for s in cells if len(s)==2)}
    faces=[s for s in cells if len(s)==3]
    z=np.array([int(s in fundamental_faces) for s in faces],dtype=np.int64)
    assert np.all((d2@z)%2==0)
    pairing=np.array([[sum(a[edges[(i,j)]]*b[edges[(j,k)]]
                         for i,j,k in fundamental_faces)%2 for b in h] for a in h])
    for a in h: assert np.all((d2.T@a)%2==0)
    return betti,pairing


In [2]:
n=4
v=lambda i,j:(i%n)*n+(j%n)
torus_faces=[]
for i in range(n):
    for j in range(n):
        a,b,c,d=v(i,j),v(i+1,j),v(i+1,j+1),v(i,j+1)
        torus_faces.extend([(a,b,c),(a,c,d)])
torus=closure(torus_faces)
torus_z={tuple(sorted(s)) for s in torus_faces}
sphere_faces=list(itertools.combinations([0,5,6,7],3))
wedge=closure([(0,1),(1,2),(2,0),(0,3),(3,4),(4,0)]+sphere_faces)
wedge_z=set(sphere_faces)
bt,pt=inspect(torus,torus_z)
bw,pw=inspect(wedge,wedge_z)
assert bt==bw==(1,2,1)
assert np.array_equal(pt,[[0,1],[1,0]])
assert np.array_equal(pw,[[0,0],[0,0]])
print('Torus Betti:',bt,'; cup pairing:',pt.tolist())
print('Wedge Betti:',bw,'; cup pairing:',pw.tolist())


Torus Betti: (1, 2, 1) ; cup pairing: [[0, 1], [1, 0]]
Wedge Betti: (1, 2, 1) ; cup pairing: [[0, 0], [0, 0]]


## 核对与反例

两个空间的同调维数相同，但环面上的两个一阶生成元杯积非零，楔和为零。因此它们不可能同伦等价。这里我们确实计算了杯积，不能仅凭可视图判断。

RP² 的整数胞腔边界是乘 2：核为零，余核为 Z/2。将系数改为 F₂ 后边界变零，二阶同调也变为一维。

In [3]:
rp2_d2=np.array([[2]],dtype=np.int64)
assert rp2_d2[0,0]==2 # integer map is injective, with cokernel Z/2
assert rank2(rp2_d2)==0
assert (1-rank2(rp2_d2),1-rank2(rp2_d2))==(1,1)
print('RP2: integral H1=Z/2, H2=0; mod-2 beta1=beta2=1')


RP2: integral H1=Z/2, H2=0; mod-2 beta1=beta2=1


## 练习与分步提示

1. 将周期网格改为 5×5，重跑全部断言；解释不变量不应依赖该细分。
2. 把 `d1@d2==0` 改成手算一个面的边界的边界，核对符号。
3. 为什么模 2 杯积矩阵看不见反对称公式中的负号？提示：先在整数上写 b∪a=−a∪b，再改变系数。

参考：Hatcher, *Algebraic Topology* (2002), 同调与上同调章节；Dey–Wang, *Computational Topology for Data Analysis* (2022)。